# VKS-turbulent: 3D coupled MHD in a box (turbulent)

Turbulent proxy (k-eps) of the VKS experiment: a Cartesian box with two
counter-rotating faces acting as impellers (`paroi_defilante`), insulating
magnetic walls, two-way coupling (`force_lorentz` + `vitesse_hydraulique`).
Sodium-like properties are taken from `share/VKS_experiment_data/sodium_properties.json`.
Runs on 2 ranks with partitioning. Smoke case only (no analytic assert).


In [ ]:
import json
from pathlib import Path

from trustutils import run

def _find_vks_data(name):
    for root in [Path.cwd(), *Path.cwd().parents]:
        candidate = root / "share" / "VKS_experiment_data" / name
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(name)

with _find_vks_data("sodium_properties.json").open() as _f:
    _props = {q["id"]: q["value"] for q in json.load(_f)["quantities"]}

dic = {
    "mu": _props["sodium_dynamic_viscosity"],
    "rho": _props["sodium_density"],
    "lambda": _props["sodium_thermal_conductivity"],
    "Cp": _props["sodium_specific_heat"],
    "eta": _props["sodium_magnetic_diffusivity"],
}

run.reset()
case = run.addCaseFromTemplate("vks_turbulent.data", "coarse", dic=dic,
                              nbProcs=2, excluNR=True)
case.partition()
run.printCases()
run.runCases()
